<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2024/2025  

<img src="../../img/portada.png" width="300"/>

---

#  Ensamble Secuencial de Modelos Predictivos  
### Primera Convocatoria – Junio 2025  

**Profesor:** Juan Galán Páez  
**Autores:**  
Carlos Martín de Prado Barragán  
Marco Padilla Gómez  

**Fecha de entrega:** 9 de Junio de 2025  

</div>

## Introducción

Este notebook está dedicado a la evaluación rigurosa del meta-modelo de ensamble secuencial implementado para tareas de regresión, aplicado al conjunto de datos sobre el precio de la vivienda.

Para ello, se utiliza la técnica de validación cruzada con 5 particiones, que permite evaluar la estabilidad y capacidad de generalización del modelo al entrenar y validar sobre diferentes subconjuntos de datos.

El meta-modelo utiliza como estimador base un árbol de decisión (`DecisionTreeRegressor`) y se preprocesan las variables mediante estandarización para mejorar el rendimiento.

La métrica principal de evaluación es el coeficiente de determinación R², que indica la proporción de variabilidad explicada por el modelo en cada partición. Finalmente, se reportan la media y desviación estándar de esta métrica para valorar la eficacia del ensamble secuencial.

## 1. Importación de librerías y configuración del entorno

En esta sección se importan todas las librerías necesarias para la evaluación mediante validación cruzada del meta-modelo:

- Se añade la ruta `../../src` para poder importar el módulo `sequential_ensemble.py` que contiene la clase `SequentialEnsembleRegressor`.
- Se importan librerías fundamentales para manejo de datos (`numpy`, `pandas`), preprocesamiento y modelado (`DecisionTreeRegressor`, `StandardScaler`, `OneHotEncoder`), para transformar las columnas(`ColumnTransformer`) y para realizar validación cruzada (`cross_val_score`, `KFold`).
- Finalmente, se importa el meta-modelo secuencial personalizado para su uso en los experimentos.

In [1]:
# Importación de librerías necesarias
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer

# Importar el meta-modelo desde nuestro módulo
import sys
sys.path.append('../../src')

from sequential_ensemble import SequentialEnsembleRegressor

## 2. Carga y preprocesamiento del dataset

- Se carga el conjunto de datos `house_prices.csv` desde la carpeta `data`.
- Se separan las variables predictoras (`X`) y la variable objetivo (`y`), que corresponde a la escala `SalePrice`.
- Se aplica estandarización a las variables predictoras utilizando `StandardScaler` para normalizar su escala y facilitar el entrenamiento del modelo y `OneHotEncoder` para codificar las variables categóricas.
- Se combinan ambas transformaciones con un `ColumnTransformer`.

In [2]:
# Cargar el dataset de Parkinson
df = pd.read_csv("../../data/house_prices.csv")

# Separar variables predictoras y variable objetivo
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

# Detectar variables categóricas (tipo objeto) y numéricas
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

# Pipeline de preprocesamiento
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

# Aplicar el preprocesamiento al conjunto de datos
X_processed = preprocessor.fit_transform(X)

## 3. Configuración del meta-modelo y validación cruzada

- Se configura la validación cruzada con 5 particiones (folds), habilitando el barajado aleatorio de los datos para garantizar particiones representativas.
- Se crea una instancia del meta-modelo secuencial `SequentialEnsembleRegressor` utilizando árboles de decisión como estimador base.
- Los hiperparámetros configurados son:
  - `n_estimators=200`: número de modelos a entrenar en el ensamble.
  - `lr=0.05`: tasa de aprendizaje que modula la contribución de cada modelo.
  - `sample_size=0.75`: proporción de muestras usadas para entrenar cada modelo base.
  - `max_depth=6`: profundidad máxima del árbol base para controlar su complejidad.
- Se establece una semilla (`random_state=42`) para asegurar la reproducibilidad de los resultados.

In [3]:
# Configurar validación cruzada con 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Crear el meta-modelo secuencial con DecisionTreeRegressor
modelo = SequentialEnsembleRegressor(
    base_estimator=DecisionTreeRegressor,
    n_estimators=200,
    lr=0.05,
    sample_size=0.75,
    est_params={"max_depth": 6},
    random_state=42
)

## 4. Ejecución de la validación cruzada y presentación de resultados

- Se ejecuta la validación cruzada usando la función `cross_val_score` con la métrica R² como criterio de evaluación.
- El modelo se entrena y evalúa en cada partición generada por el objeto `KFold`.
- Se muestran los valores de R² obtenidos en cada fold para analizar la variabilidad del rendimiento.
- Se calcula y presenta la media y desviación estándar de los valores R², que representan la eficacia promedio y la estabilidad del modelo respectivamente.

In [4]:
# Ejecutar validación cruzada con scoring R²
scores = cross_val_score(modelo, X_processed, y, cv=kf, scoring='r2')

# Mostrar resultados
print(f"R2 scores por fold: {scores}")
print(f"Media R2: {scores.mean():.4f}")
print(f"Desviación estándar R2: {scores.std():.4f}")

R2 scores por fold: [0.70771433 0.7785821  0.74006385 0.78411637 0.79445206]
Media R2: 0.7610
Desviación estándar R2: 0.0324


## 5. Conclusiones

- El meta-modelo secuencial entrenado con árboles de decisión ha mostrado un rendimiento sólido y consistente en el conjunto de datos de house_price.
- Los valores de R² obtenidos en las cinco particiones de la validación cruzada oscilan entre aproximadamente 0.70 y 0.80, indicando una buena capacidad predictiva.
- La media del coeficiente de determinación R² es de aproximadamente 0.75, lo que significa que el modelo explica aproximadamente el 75% de la variabilidad en los datos de prueba.
- La baja desviación estándar (0.03 aprox.) evidencia una alta estabilidad y robustez del modelo frente a diferentes particiones de los datos.
- Estos resultados refuerzan la eficacia del enfoque de ensamble secuencial para mejorar la precisión y generalización en problemas de regresión complejos.